# Sort ellipsoid objects by their orientiation with respect to the central vein
The small vessel channel (anti-CD105 staining with Alexa546) was segmented with Imaris or TubeMap and exported as binary images.
A Fiji macro counted all partical in each slice which have an elliptical shape (sphericity between 0.2-0.5)
Here, the exported data will be summarized with respect to the orientation to the central vein

Input: csv-lists with the properties of the elliptical objects found in the CD105 segmentation

Output: txt-files summarizing the properties of the elliptical objects according to their orientation with respect to the central vein

In [1]:
# 1. Import all required libraries
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import pandas as pd

In [2]:
# 3. folder to be processed
path = '*_z-y.csv'
save_file_as = 'Ellipsoids_zy-plane.txt'

list_filenames = list()
list_properties = list()

In [3]:
# 4. loop over the folder to process all images
for file in glob.glob(path):
    # Get data
    filename = os.path.abspath(file).split(".")[0]
    df = pd.read_csv(file)
    # Column 9: Angle (with respect to x-axis of image) , Column 10: Circularity
    # Column 14: Roundness (i.e. 1/aspect ratio), Column 15: solidity (area/convex area)
    
    # Extract column #9
    raw_angles = df.iloc[:, 9]  # Python is 0-indexed, so column 9 corresponds to index 8
    
    # Add a new column based on the condition
    df['aligned_angles'] = np.where(raw_angles > 90, raw_angles - 180, raw_angles)
    
   # Vessels parallel to central vein, angle between +20 & -20°
    parallel_df = df[(df['aligned_angles'] >= -20) & (df['aligned_angles'] <= 20)]
    
    # Determine the number of objects (rows) after filtering
    nr_objects_parallel = len(parallel_df)
    
    # Vessels perpendicular to central vein, angle between +70 to -70
    perpendicular_df = df[(df['aligned_angles'] <= -70) | (df['aligned_angles'] >= 70)]
    
    # Determine the number of objects (rows) after filtering
    nr_objects_perpendicular = len(perpendicular_df)
    
    # Calculate mean, standard deviation, and other statistics of the filtered data
    mean_parallel = parallel_df['aligned_angles'].mean()
    stdev_parallel = parallel_df['aligned_angles'].std()
    min_parallel = parallel_df['aligned_angles'].min()
    max_parallel = parallel_df['aligned_angles'].max()
    
    mean_perpendicular = perpendicular_df['aligned_angles'].mean()
    stdev_perpendicular = perpendicular_df['aligned_angles'].std()
    min_perpendicular = perpendicular_df['aligned_angles'].min()
    max_perpendicular = perpendicular_df['aligned_angles'].max()
    
    props_parallel = [nr_objects_parallel, mean_parallel, stdev_parallel, min_parallel, max_parallel]
    props_perpendicular = [nr_objects_perpendicular, mean_perpendicular, stdev_perpendicular, min_perpendicular, max_perpendicular]
    
    # Append all parameter to a growing list
    list_filenames.append(str(filename))
    list_properties.append(props_parallel + props_perpendicular)

    header = 'Filename\t# Objects parallel\tAngle mean\tstdev\tmin\tmax\t# Objects perpendicular\tAngle mean\tstdev\tmin\tmax'

    # save results as txt file
    np.savetxt(
        save_file_as,
        np.vstack(
            [
                list_filenames,
                np.array(list_properties).T
            ],
        ).T,
        delimiter='\t', fmt="%s", header=header
    )
    